# MNIST Digit Classifier — Building a CNN

**What you'll learn:** how a convolutional neural network processes an image layer by layer, how to write a training loop, and how to diagnose what went wrong.

**Estimated time:** 60–90 min &nbsp;|&nbsp; **Runtime:** GPU (Runtime -> Change runtime type -> GPU)

---

### Map the journey before you start

A 28x28 grayscale image enters the network. Your job is to classify it as one of 10 digits (0–9). Between input and output, the network will:

1. Run two convolution + pooling stages to extract features
2. Flatten the result into a 1D vector
3. Pass it through a linear layer to produce 10 class scores

? **Before writing any code:** trace the dimensions yourself.
- Input: `(1, 28, 28)` — 1 channel, 28x28 pixels
- After conv1 (16 filters, 3x3, padding=1): `(16, 28, 28)` — padding preserves size
- After max-pool (2x2): `(16, ?, ?)` — halves spatial dims
- After conv2 (32 filters, 3x3, padding=1): `(32, ?, ?)`
- After max-pool: `(32, ?, ?)`
- After flatten: `(?)` — what's this number?

Write your answers before running. The final flat size is the input to `self.fc`.


## 1  Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 2  Load MNIST

In [ ]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST global mean and std
])

train_ds = datasets.MNIST('.', train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST('.', train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

print(f"Training images: {len(train_ds)}")
print(f"Test images:     {len(test_ds)}")

In [ ]:
# Look at a batch
xb, yb = next(iter(train_dl))
print("Batch shape:", xb.shape)   # (128, 1, 28, 28)
print("Label shape:", yb.shape)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(xb[i, 0], cmap='gray')
    ax.set_title(str(yb[i].item()))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3  Build the network — step by step

Now build the CNN. Fill in the architecture based on the dimension trace you did at the top.

The key question: **what value goes in `nn.Linear(???, 10)`?**

Trace it: after two conv+pool stages on a 28x28 input, the spatial size is `7x7`. With 32 channels, the flat vector is `32 x 7 x 7 = ?`


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # conv1: 1 input channel -> 16 feature maps, 3x3 kernel, padding=1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)

        # conv2: 16 -> 32 feature maps, same kernel
        # YOUR CODE HERE
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        # Fully connected: flatten -> 10 classes
        # What is the flat size after two 2x2 max-pools on a 28x28 image?
        # YOUR CODE HERE — replace ??? with the correct number
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        # Conv1 -> ReLU -> 2x2 max-pool
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)

        # Conv2 -> ReLU -> 2x2 max-pool
        # YOUR CODE HERE — same pattern as above
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)

        # Flatten all dimensions except batch, then classify
        x = x.flatten(1)
        return self.fc(x)

model = Net().to(device)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### Verify your dimensions

Before training, pass a single fake image through the network to confirm the shapes are correct. If there's a size mismatch, you'll get an error here — much better than finding out mid-training.


In [ ]:
with torch.no_grad():
    dummy = torch.zeros(1, 1, 28, 28).to(device)   # one fake image
    out = model(dummy)
    print("Output shape:", out.shape)               # should be (1, 10)
    print("OK Dimensions OK!" if out.shape == (1, 10) else "X Shape is wrong — recheck your network")

## 4  Write the training loop

The training loop does this on every batch:
1. **Zero gradients** — clear leftover gradients from the last step
2. **Forward pass** — run the batch through the model
3. **Compute loss** — compare predictions to true labels (cross-entropy)
4. **Backward pass** — compute gradients via backpropagation
5. **Update weights** — optimizer takes one step

Fill in steps 1–5 below.


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, opt):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        # YOUR CODE HERE — fill in the 5 steps
        opt.zero_grad()                          # 1. zero gradients
        preds = model(xb)                        # 2. forward pass
        loss = F.cross_entropy(preds, yb)        # 3. compute loss
        loss.backward()                          # 4. backward
        opt.step()                               # 5. update weights

        total_loss += loss.item()
    return total_loss / len(loader)

# Quick sanity check: run one epoch
loss = train_epoch(model, train_dl, opt)
print(f"First epoch avg loss: {loss:.4f}")
print("(should be around 0.3–0.5 after the first pass)")

## 5  Train for real

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(loader.dataset)

history = []
for epoch in range(5):
    loss = train_epoch(model, train_dl, opt)
    acc  = evaluate(model, test_dl)
    history.append((loss, acc))
    print(f"Epoch {epoch+1}: loss={loss:.4f}  test_acc={acc:.4f}")

In [ ]:
losses, accs = zip(*history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses, marker='o'); ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.grid(alpha=0.3)
ax2.plot(accs,   marker='o', color='green'); ax2.set_title('Test Accuracy'); ax2.set_xlabel('Epoch'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final test accuracy: {accs[-1]:.4f} ({accs[-1]*100:.2f}%)")

? **Reflect:**
- What was your final accuracy?
- The network has ~21,000 parameters. A baseline "always guess the most common class" gets ~10%. How much better did you do?
- Did loss go down smoothly? What would it mean if it didn't?


## 6  Diagnose: what does it get wrong?

In [ ]:
import numpy as np

model.eval()
wrong_imgs, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for xb, yb in test_dl:
        preds = model(xb.to(device)).argmax(1).cpu()
        mask = preds != yb
        wrong_imgs.extend(xb[mask])
        wrong_true.extend(yb[mask].tolist())
        wrong_pred.extend(preds[mask].tolist())
        if len(wrong_imgs) >= 16:
            break

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i >= len(wrong_imgs): break
    ax.imshow(wrong_imgs[i][0], cmap='gray')
    ax.set_title(f"✓{wrong_true[i]}
✗{wrong_pred[i]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Misclassified digits (✓=true, ✗=predicted)')
plt.tight_layout()
plt.show()

? **Study the failures.** Which digit pairs does the model confuse most? (4 vs 9? 3 vs 8?) Can you see why — do those digits look similar?

Now check the confusion matrix for a systematic view:


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

all_true, all_pred = [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        all_true.extend(yb.tolist())
        all_pred.extend(model(xb.to(device)).argmax(1).cpu().tolist())

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Which class has the most errors?
errors_per_class = cm.sum(axis=1) - cm.diagonal()
print("Errors per digit:", dict(enumerate(errors_per_class)))

## 7  Improve it yourself

Pick **one** of these and verify it actually helps:

1. **Add dropout** (`nn.Dropout(0.25)` before the linear layer) — does it improve or worsen accuracy? Why?
2. **Add a third conv layer** (64 filters after the second pool) — remember to update the fc input size.
3. **Try SGD instead of Adam** (lr=0.01, momentum=0.9) — is Adam really better here?
4. **Train longer** (10 epochs) — where does accuracy plateau?

Report what you tried and whether it helped. Understanding why a change didn't help is just as valuable as finding one that did.
